In [1]:
%pip -q install pyarrow scikit-learn

import os
import gc
import time
import random

from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from IPython.display import display

from google.colab import drive
drive.mount("/content/drive")

# ============================================================
# CONFIGURATION
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

ROOT = Path("/content/drive/MyDrive/bt4222/JDsearch")

STEP1A = ROOT / "step1a_history"
STEP1B = ROOT / "step1b_query_item" / "full"

SAMPLED = ROOT / "user_subset.parquet"
METADATA = (
    ROOT / "step2_baselines"
    / "candidate_metadata_compact.parquet"
)

OUT = ROOT / "step2_baselines"
OUT.mkdir(parents=True, exist_ok=True)

# Experiment switches
QUICK_RUN = False
USE_METADATA = True

LIMITS = {
    "train": 1000,
    "val": 300,
    "test": 300
}

MAX_HISTORY = 200
MAX_VOCAB = 250_000

EMB = 64
HIDDEN = 64
BATCH = 32

EPOCHS = 2 if QUICK_RUN else 12
NEG_PER_POS = 20

print("Device:", DEVICE)
if DEVICE != "cuda":
    print("WARNING: CPU training will be much slower. In Colab select Runtime > Change runtime type > T4 GPU, then restart.")
print("Quick run:", QUICK_RUN)
print("Metadata:", USE_METADATA)
print("Data directory:", ROOT)

Mounted at /content/drive
Device: cuda
Quick run: False
Metadata: True
Data directory: /content/drive/MyDrive/bt4222/JDsearch


In [2]:
# ============================================================
# EARLY SESSION FILTERING
# ============================================================

source = pd.read_parquet(
    SAMPLED,
    columns=["split"]
)

# session_id = original row number
source["session_id"] = np.arange(len(source), dtype=np.int64)

source["split"] = (
    source["split"]
    .astype(str)
    .str.lower()
    .replace({"validation": "val", "valid": "val"})
)

if QUICK_RUN:
    selected = pd.concat(
        [
            source[source["split"] == s].head(LIMITS[s])
            for s in ["train", "val", "test"]
        ],
        ignore_index=True
    )
else:
    selected = source.copy()

selected_sids = set(selected["session_id"])

print("Selected sessions:", len(selected))
print(selected["split"].value_counts())

del source
gc.collect()

Selected sessions: 85864
split
train    60104
val      12880
test     12880
Name: count, dtype: int64


16

In [3]:
import pyarrow.dataset as ds

def load_sessions(path, columns=None):
    path = Path(path)

    if QUICK_RUN:
        # Arrow filters before Pandas materialization.
        dataset = ds.dataset(str(path), format="parquet")

        table = dataset.to_table(
            columns=columns,
            filter=ds.field("session_id").isin(
                sorted(selected_sids)
            )
        )

        return table.to_pandas()

    return pd.read_parquet(path, columns=columns)


paths = {
    "full": STEP1A / "history_full.parquet",
    "hu": STEP1A / "history_user_features.parquet",
    "hc": STEP1A / "history_candidate_features.parquet",
    "hg": STEP1A / "history_group_features.parquet",
    "qi_train": STEP1B / "query_item_train.parquet",
    "qi_val": STEP1B / "query_item_val.parquet",
    "qi_test": STEP1B / "query_item_test.parquet",
}

if USE_METADATA:
    paths["metadata"] = METADATA

for name, path in paths.items():
    print(name, path.exists(), path)

missing = [str(p) for p in paths.values() if not p.exists()]

if missing:
    raise FileNotFoundError("\n".join(missing))

full True /content/drive/MyDrive/bt4222/JDsearch/step1a_history/history_full.parquet
hu True /content/drive/MyDrive/bt4222/JDsearch/step1a_history/history_user_features.parquet
hc True /content/drive/MyDrive/bt4222/JDsearch/step1a_history/history_candidate_features.parquet
hg True /content/drive/MyDrive/bt4222/JDsearch/step1a_history/history_group_features.parquet
qi_train True /content/drive/MyDrive/bt4222/JDsearch/step1b_query_item/full/query_item_train.parquet
qi_val True /content/drive/MyDrive/bt4222/JDsearch/step1b_query_item/full/query_item_val.parquet
qi_test True /content/drive/MyDrive/bt4222/JDsearch/step1b_query_item/full/query_item_test.parquet
metadata True /content/drive/MyDrive/bt4222/JDsearch/step2_baselines/candidate_metadata_compact.parquet


In [4]:
# ============================================================
# JOIN FEATURE ENGINEERING OUTPUTS
# ============================================================

def norm_id(x):
    if pd.isna(x):
        return None

    s = str(x).strip()

    if s.endswith(".0") and s[:-2].isdigit():
        s = s[:-2]

    return s


def check_unique(d, keys, name):
    n = d.duplicated(keys).sum()
    print(f"{name} duplicate keys: {n:,}")


qi_parts = []

for s in ["train", "val", "test"]:
    t = load_sessions(paths[f"qi_{s}"])

    t = t.rename(columns={"candidate_id": "wid"})
    t["wid"] = t["wid"].map(norm_id)

    assert (
        t["split"].astype(str).str.lower() == s
    ).all(), f"Split mismatch in {s}"

    qi_parts.append(t)

qi = pd.concat(qi_parts, ignore_index=True)
del qi_parts

original_rows = len(qi)

check_unique(qi, ["session_id", "wid"], "Step 1b")

# ------------------------------------------------------------
# Candidate history features
# ------------------------------------------------------------

hc = load_sessions(paths["hc"])
hc["wid"] = hc["wid"].map(norm_id)

check_unique(hc, ["session_id", "wid"], "Step 1a candidate")

hc = hc.rename(columns={
    c: f"hc__{c}"
    for c in hc.columns
    if c not in ["session_id", "wid", "label", "split"]
})

df = qi.merge(
    hc,
    on=["session_id", "wid"],
    how="left",
    suffixes=("_qi", "_hc"),
    indicator=True,
    validate="one_to_one"
)

print("Candidate join unmatched:",
      (df["_merge"] != "both").sum())

print("Label mismatches:",
      ((df["label_qi"] != df["label_hc"]) &
       df["label_hc"].notna()).sum())

print("Split mismatches:",
      ((df["split_qi"] != df["split_hc"]) &
       df["split_hc"].notna()).sum())

df = (
    df.drop(columns=["_merge", "label_hc", "split_hc"])
      .rename(columns={
          "label_qi": "label",
          "split_qi": "split"
      })
)

del hc, qi
gc.collect()

# ------------------------------------------------------------
# User history features
# ------------------------------------------------------------

hu = load_sessions(paths["hu"])
check_unique(hu, ["session_id"], "Step 1a user")

hu = hu.drop(columns=["split"])

hu = hu.rename(columns={
    c: f"hu__{c}"
    for c in hu.columns
    if c != "session_id"
})

df = df.merge(
    hu,
    on="session_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

print("Missing user joins:",
      (df["_merge"] != "both").sum())

df = df.drop(columns="_merge")

del hu
gc.collect()

# ------------------------------------------------------------
# Brand/shop/category history features
# ------------------------------------------------------------

hg = load_sessions(paths["hg"])
hg["wid"] = hg["wid"].map(norm_id)

check_unique(hg, ["session_id", "wid"], "Step 1a group")

hg = hg.rename(columns={
    c: f"hg__{c}"
    for c in hg.columns
    if c not in ["session_id", "wid"]
})

df = df.merge(
    hg,
    on=["session_id", "wid"],
    how="left",
    validate="one_to_one",
    indicator=True
)

print("Missing group joins:",
      (df["_merge"] != "both").sum())

df = df.drop(columns="_merge")

del hg
gc.collect()

print("\nOriginal candidate rows:", original_rows)
print("Joined candidate rows:", len(df))
print("Rows lost:", original_rows - len(df))

display(
    df.groupby("split").agg(
        sessions=("session_id", "nunique"),
        candidates=("wid", "size"),
        positives=("label", lambda x: (x > 0).sum())
    )
)

Step 1b duplicate keys: 0
Step 1a candidate duplicate keys: 0
Candidate join unmatched: 0
Label mismatches: 0
Split mismatches: 0
Step 1a user duplicate keys: 0
Missing user joins: 0
Step 1a group duplicate keys: 0
Missing group joins: 0

Original candidate rows: 7665446
Joined candidate rows: 7665446
Rows lost: 0


,sessions,candidates,positives
split,,,
test,12880,1147673,23929
train,60104,5360982,111997
val,12880,1156791,24309


In [5]:
# ============================================================
# JOIN INTEGRITY CHECKS
# ============================================================

print("All candidate rows retained:", len(df) == original_rows)

print(
    "Sessions with multiple splits:",
    (df.groupby("session_id")["split"].nunique() > 1).sum()
)

print(
    "Invalid labels:",
    (~df["label"].isin([0, 1, 2, 3])).sum()
)

print(
    "Missing candidate IDs:",
    df["wid"].isna().sum()
)

print(
    "Duplicate session-product pairs:",
    df.duplicated(["session_id", "wid"]).sum()
)

print("\nFeature-family counts:")

for prefix in ["hc__", "hg__", "hu__"]:
    print(
        prefix,
        sum(c.startswith(prefix) for c in df.columns)
    )

All candidate rows retained: True
Sessions with multiple splits: 0
Invalid labels: 0
Missing candidate IDs: 0
Duplicate session-product pairs: 0

Feature-family counts:
hc__ 9
hg__ 15
hu__ 17


In [6]:
# ============================================================
# NUMERICAL FEATURE PREPARATION
# ============================================================

EXCLUDE = {
    "session_id",
    "wid",
    "label",
    "split",
    "candidate_pos",
    "candidate_id",
    "has_purchase"
}

FEATURES = []

for col in df.columns:

    if col in EXCLUDE:
        continue

    if not pd.api.types.is_numeric_dtype(df[col]):
        continue

    lc = col.lower()

    if any(x in lc for x in [
        "position",
        "rank",
        "label",
        "has_purchase",
        "target",
        "conversion_rate",
        "purchase_rate",
        "click_rate",
        "cart_rate"
    ]):
        continue

    FEATURES.append(col)

print("Numerical predictors:", len(FEATURES))
print(FEATURES)

train_mask = df["split"].eq("train")

med = (
    df.loc[train_mask, FEATURES]
      .median()
      .fillna(0)
)

scaler = StandardScaler()

scaler.fit(
    df.loc[train_mask, FEATURES]
      .fillna(med)
      .astype("float32")
)

X = np.empty(
    (len(df), len(FEATURES)),
    dtype="float32"
)

for start in range(0, len(df), 100_000):

    end = min(start + 100_000, len(df))

    block = (
        df.iloc[start:end][FEATURES]
          .fillna(med)
          .astype("float32")
    )

    X[start:end] = scaler.transform(block)

print("Feature matrix:", X.shape)
print("Memory MB:", round(X.nbytes / 1e6, 2))

assert np.isfinite(X).all()

Numerical predictors: 53
['has_metadata', 'query_name_overlap', 'query_brand_overlap', 'query_cate_overlap', 'bm25_query_name', 'name_length', 'has_brand', 'shop_is_missing', 'cate_depth', 'brand_product_count', 'shop_product_count', 'cate4_product_count', 'hc__prod_n_click', 'hc__prod_n_cart', 'hc__prod_n_follow', 'hc__prod_n_order', 'hc__prod_days_since_click', 'hc__prod_days_since_cart', 'hc__prod_days_since_order', 'hc__prod_days_since_any', 'hc__prod_seen_before', 'hu__n_click_7d', 'hu__n_cart_90d', 'hu__n_follow_90d', 'hu__n_order_rows_365d', 'hu__n_checkouts_365d', 'hu__n_order_products_365d', 'hu__has_click', 'hu__days_since_last_click', 'hu__has_cart', 'hu__days_since_last_cart', 'hu__has_order', 'hu__days_since_last_order', 'hu__days_since_last_action', 'hu__in_session', 'hu__n_actions_7d', 'hu__intent_share_7d', 'hu__history_len_full', 'hg__brand_n_click', 'hg__brand_n_cart', 'hg__brand_n_order', 'hg__brand_days_since_any', 'hg__brand_days_since_order', 'hg__shop_n_click', '

In [7]:
# ============================================================
# HISTORIES AND ITEM VOCABULARY
# ============================================================

full = load_sessions(
    paths["full"],
    columns=[
        "session_id",
        "split",
        "history_wid_list",
        "history_type_list",
        "history_time_list"
    ]
)

def parse_list(x):
    if isinstance(x, (list, tuple, np.ndarray)):
        return list(x)

    if x is None or (isinstance(x, float) and np.isnan(x)):
        return []

    return str(x).split("_") if str(x) else []


H = {}

for row in full.itertuples(index=False):

    sid = int(row.session_id)

    wids = parse_list(row.history_wid_list)
    acts = parse_list(row.history_type_list)
    times = parse_list(row.history_time_list)

    if not (
        len(wids) == len(acts)
        and len(times) == len(wids) + 1
    ):
        raise ValueError(f"History alignment error: {sid}")

    H[sid] = (
        wids[-MAX_HISTORY:],
        acts[-MAX_HISTORY:],
        times[-(MAX_HISTORY + 1):]
    )

print("Loaded histories:", len(H))

train_sids = set(
    selected.loc[
        selected["split"] == "train",
        "session_id"
    ]
)

# POPULARITY BASELINE

from collections import Counter

HISTORY_LONG = (
    ROOT / "step1a_history" / "history_long.parquet"
)

hist = pd.read_parquet(
    HISTORY_LONG,
    columns=["session_id", "wid", "type", "age_sec"]
)

# Training users only
hist = hist[hist["session_id"].isin(train_sids)].copy()

# Same 1-second leakage guard as Step 3b
hist = hist[hist["age_sec"] > 1].copy()

hist["wid"] = hist["wid"].map(norm_id)
hist["type"] = hist["type"].astype(str)

hist = hist.dropna(subset=["wid"])

# ------------------------------------------------------------
# 1. Distinct training-user popularity
# ------------------------------------------------------------

user_product = hist[
    ["session_id", "wid"]
].drop_duplicates()

pop_product_users = Counter(
    user_product["wid"].value_counts().to_dict()
)

# ------------------------------------------------------------
# 2. Historical order frequency
# ------------------------------------------------------------

orders = hist[hist["type"] == "ORD"]

pop_product_ord = Counter(
    orders["wid"].value_counts().to_dict()
)

# ------------------------------------------------------------
# 3. Own-history contributions for training evaluation
# ------------------------------------------------------------

own_products = set(
    zip(user_product["session_id"], user_product["wid"])
)

own_orders = Counter(
    zip(orders["session_id"], orders["wid"])
)

print("Training history rows:", f"{len(hist):,}")
print("Products with popularity:", f"{len(pop_product_users):,}")
print("Products with orders:", f"{len(pop_product_ord):,}")

del hist, user_product, orders
gc.collect()

# Historical interaction popularity
popularity_counts = Counter()

for sid in train_sids:
    popularity_counts.update(
        norm_id(w) for w in H[int(sid)][0]
    )

# Item vocabulary based on training histories and
# training candidate identities
vocab_counts = Counter(popularity_counts)

vocab_counts.update(
    df.loc[df["split"] == "train", "wid"].tolist()
)

vocab = {
    w: i + 2
    for i, (w, _) in enumerate(
        (pair for pair in vocab_counts.most_common()
         if pair[0] is not None)
    )
    if i < MAX_VOCAB - 2
}

PAD_IDX = 0
UNK_IDX = 1

ACT = {
    "CLICK": 1,
    "CART": 2,
    "ORD": 3,
    "FLW": 4
}

item_idx = (
    df["wid"]
    .map(lambda w: vocab.get(w, UNK_IDX))
    .to_numpy(dtype="int32")
)

print("Item vocabulary:", len(vocab))

for s in ["train", "val", "test"]:
    mask = df["split"].eq(s).to_numpy()

    print(
        s,
        "Unknown item rate:",
        round((item_idx[mask] == UNK_IDX).mean(), 5)
    )

del full
gc.collect()

Loaded histories: 85864
Training history rows: 9,218,252
Products with popularity: 2,580,944
Products with orders: 934,838
Item vocabulary: 249998
train Unknown item rate: 0.79314
val Unknown item rate: 0.81516
test Unknown item rate: 0.8135


0

In [8]:
# ============================================================
# SHARED POPULARITY BASELINE
# ============================================================

from collections import Counter

# Number of distinct training users interacting with each product
pop_product_users = Counter()

# Number of historical orders for each product
pop_product_ord = Counter()

# Per-training-user contributions for leave-self-out
own_products = {}
own_orders = {}

for sid in train_sids:

    wids, actions, times = H[int(sid)]

    # Reconstruct age of each historical action
    gaps = np.asarray(times, dtype=np.float64)

    ages = np.cumsum(gaps[::-1])[::-1][:-1]

    # Exclude events within 1 second of final query
    valid = ages > 1

    products = [
        norm_id(w)
        for w, keep in zip(wids, valid)
        if keep
    ]

    orders = [
        norm_id(w)
        for w, a, keep in zip(wids, actions, valid)
        if keep and a == "ORD"
    ]

    # Each user contributes at most once to distinct-user count
    unique_products = set(products)

    pop_product_users.update(unique_products)
    pop_product_ord.update(orders)

    own_products[int(sid)] = unique_products
    own_orders[int(sid)] = Counter(orders)

print("Products with distinct-user popularity:",
      len(pop_product_users))

print("Products with historical orders:",
      len(pop_product_ord))

Products with distinct-user popularity: 1779705
Products with historical orders: 571094


In [9]:
# ============================================================
# FOUR-EMBEDDING METADATA LOOKUP
# ============================================================

ATTR_COLS = [
    "brand_id",
    "shop_id",
    "cate_id_4"
]

candidate_attrs = np.ones(
    (len(df), len(ATTR_COLS)),
    dtype=np.int32
)

attr_card = {}

if USE_METADATA:

    required_wids = set(df["wid"].dropna())

    pf = pq.ParquetFile(METADATA)

    meta_parts = []

    for batch in pf.iter_batches(
        batch_size=100_000,
        columns=["wid"] + ATTR_COLS
    ):

        part = batch.to_pandas()

        part["wid"] = part["wid"].map(norm_id)

        part = part[
            part["wid"].isin(required_wids)
        ]

        if not part.empty:
            meta_parts.append(part)

        del part

    if meta_parts:
        meta = pd.concat(meta_parts, ignore_index=True)
    else:
        meta = pd.DataFrame(columns=["wid"] + ATTR_COLS)

    duplicate_meta = meta.duplicated("wid").sum()

    print("Duplicate metadata WIDs:", duplicate_meta)

    meta = meta.drop_duplicates("wid")

    print("Matched metadata products:", len(meta))
    print("Candidate products:", len(required_wids))

    # Build categorical lookup tables
    for j, col in enumerate(ATTR_COLS):

        vals = meta[col].map(norm_id)

        valid = (
            vals.notna()
            & ~vals.isin(["-1", "", "nan", "None"])
        )

        categories = pd.Index(vals[valid].unique())

        mapping = {
            v: i + 2
            for i, v in enumerate(categories)
        }

        attr_card[col] = len(mapping) + 2

        lookup = pd.Series(
            vals.map(
                lambda v: mapping.get(v, UNK_IDX)
            ).to_numpy(dtype="int32"),
            index=meta["wid"]
        )

        candidate_attrs[:, j] = (
            df["wid"]
            .map(lookup)
            .fillna(UNK_IDX)
            .to_numpy(dtype="int32")
        )

        del lookup, mapping, categories

    print("Attribute vocabulary sizes:", attr_card)

    for j, col in enumerate(ATTR_COLS):
        print(
            col,
            "Unknown rate:",
            round(
                (candidate_attrs[:, j] == UNK_IDX).mean(),
                5
            )
        )

    del meta, meta_parts, required_wids
    gc.collect()

else:

    print("Metadata disabled: item-ID embeddings only")

Duplicate metadata WIDs: 0
Matched metadata products: 4829298
Candidate products: 4859472
Attribute vocabulary sizes: {'brand_id': 145357, 'shop_id': 219070, 'cate_id_4': 6505}
brand_id Unknown rate: 0.00456
shop_id Unknown rate: 0.01423
cate_id_4 Unknown rate: 0.00456


In [10]:
# ============================================================
# RANKING METRICS
# ============================================================

GROUP_KEY = "session_id"
LABEL = "label"
EVAL_K = [5, 10]

def rank_metrics(d, score, ks=EVAL_K, seed=0):

    # Construct evaluation dataframe
    t = pd.DataFrame({
        "g": d[GROUP_KEY].to_numpy(),
        "y": d[LABEL].to_numpy(),
        "s": np.asarray(score, dtype="float64"),
        "wid": d["wid"].map(norm_id).to_numpy()
    })

    # Restore original candidate order before tie-breaking
    canonical = (
        df[["session_id", "wid", "candidate_pos"]]
        .drop_duplicates(["session_id", "wid"])
    )

    t = t.merge(
        canonical,
        left_on=["g", "wid"],
        right_on=["session_id", "wid"],
        how="left",
        validate="one_to_one"
    )

    if t["candidate_pos"].isna().any():
        raise ValueError(
            "Missing candidate positions during ranking evaluation"
        )

    t = t.sort_values(
        ["g", "candidate_pos"],
        kind="stable"
    ).reset_index(drop=True)

    # Generate deterministic tie-breaking after canonical ordering
    rng = np.random.default_rng(seed)
    t["tie"] = rng.random(len(t))

    # Rank candidates by model score
    t = t.sort_values(
        ["g", "s", "tie"],
        ascending=[True, False, True],
        kind="stable"
    )

    t["r"] = t.groupby("g").cumcount() + 1
    t["gain"] = 2.0 ** t["y"] - 1
    t["disc"] = 1 / np.log2(t["r"] + 1)

    ideal = t.sort_values(
        ["g", "y"],
        ascending=[True, False],
        kind="stable"
    ).copy()

    ideal["r"] = ideal.groupby("g").cumcount() + 1
    ideal["disc"] = 1 / np.log2(ideal["r"] + 1)

    out = {}

    for k in ks:

        dcg = (
            (t["gain"] * t["disc"])
            .where(t["r"] <= k, 0)
            .groupby(t["g"])
            .sum()
        )

        idcg = (
            (ideal["gain"] * ideal["disc"])
            .where(ideal["r"] <= k, 0)
            .groupby(ideal["g"])
            .sum()
        )

        out[f"ndcg@{k}"] = (dcg / idcg).mean()

    first = (
        t[t["y"] > 0]
        .groupby("g")["r"]
        .min()
    )

    out["mrr"] = (1 / first).mean()

    buy = t[t["y"] == 3]

    if len(buy):

        out["purchase_hit@10"] = (
            buy.groupby("g")["r"].min() <= 10
        ).mean()

        out["n_purchase_searches"] = buy["g"].nunique()

    out["n_searches"] = t["g"].nunique()

    return out

In [11]:
# ============================================================
# PRE-ENCODE HISTORIES AND CANDIDATE TARGETS ONCE
# Avoid repeated pandas .iloc and string conversions in every batch.
# ============================================================
import math
from torch.utils.data import Sampler

labels_np = df["label"].to_numpy(dtype=np.int64, copy=True)
wid_np = df["wid"].to_numpy(copy=False)
# All histories are already capped at MAX_HISTORY in H.
H_encoded = {}
for sid, (wids, acts, times) in H.items():
    ww = np.fromiter((vocab.get(norm_id(w), UNK_IDX) for w in wids), dtype=np.int64, count=len(wids))
    aa = np.fromiter((ACT.get(str(a), 0) for a in acts), dtype=np.int64, count=len(acts))
    tt = np.asarray(times[:-1], dtype=np.float32)
    if len(tt) != len(ww):
        raise ValueError(f"History alignment failure for {sid}")
    tt = np.log1p(np.maximum(tt, 0))
    H_encoded[int(sid)] = (ww, aa, tt)

session_rows = {s: [] for s in ["train", "val", "test"]}
for sid, inds in df.groupby("session_id", sort=False).indices.items():
    ix = np.asarray(inds, dtype=np.int32)
    session_rows[str(df["split"].iat[int(ix[0])])].append((int(sid), ix))
for s in session_rows:
    print(s, "sessions", len(session_rows[s]), "candidate rows", sum(len(ix) for _, ix in session_rows[s]))

class Sessions(Dataset):
    def __init__(self, split, full_slate=False):
        self.split=split
        self.rows=session_rows[split]
        self.full_slate=full_slate
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        sid, idx=self.rows[i]
        if self.split == "train" and not self.full_slate and NEG_PER_POS is not None:
            y=labels_np[idx]
            pos=idx[y>0]; neg=idx[y==0]
            # Independent reproducible negative draw per access; no pandas indexing.
            take=min(len(neg), max(1,len(pos))*NEG_PER_POS)
            if take < len(neg):
                chosen=np.random.choice(neg,size=take,replace=False)
                idx=np.concatenate((pos,chosen))
        return sid,idx,*H_encoded[sid]

def collate(batch):
    B=len(batch);T=max(1,max(len(v[2]) for v in batch));K=max(len(v[1]) for v in batch)
    w=np.zeros((B,T),np.int64);a=np.zeros((B,T),np.int64);gap=np.zeros((B,T),np.float32)
    cand=np.zeros((B,K),np.int64);attrs=np.zeros((B,K,len(ATTR_COLS)),np.int64)
    feat=np.zeros((B,K,X.shape[1]),np.float32);lab=np.full((B,K),-100,np.int64)
    idxs=[];sids=[]
    for j,(sid,idx,ww,aa,tt) in enumerate(batch):
        sids.append(sid);idxs.append(idx);L=len(ww);N=len(idx)
        if L:w[j,:L]=ww;a[j,:L]=aa;gap[j,:L]=tt
        cand[j,:N]=item_idx[idx];attrs[j,:N]=candidate_attrs[idx];feat[j,:N]=X[idx];lab[j,:N]=labels_np[idx]
    return {k:torch.from_numpy(v) for k,v in dict(hist_wid=w,hist_act=a,hist_gap=gap,cand=cand,attrs=attrs,feat=feat,label=lab).items()},idxs,sids

class BucketBatchSampler(Sampler):
    # Group sessions of similar history/candidate length to reduce padding.
    def __init__(self, dataset, batch_size, shuffle=False):
        self.ds=dataset;self.bs=batch_size;self.shuffle=shuffle
    def __iter__(self):
        ids=np.arange(len(self.ds))
        if self.shuffle:np.random.shuffle(ids)
        pool=max(self.bs*40,self.bs)
        for st in range(0,len(ids),pool):
            block=ids[st:st+pool]
            block=sorted(block,key=lambda j:(len(H_encoded[self.ds.rows[j][0]][0]),len(self.ds.rows[j][1])))
            batches=[block[k:k+self.bs] for k in range(0,len(block),self.bs)]
            if self.shuffle:random.shuffle(batches)
            for b in batches:yield b
    def __len__(self):return math.ceil(len(self.ds)/self.bs)

def loader(split,shuffle=False,full_slate=False):
    dataset=Sessions(split,full_slate=full_slate)
    return DataLoader(dataset,batch_sampler=BucketBatchSampler(dataset,BATCH,shuffle),collate_fn=collate,num_workers=0,pin_memory=(DEVICE=='cuda'))

print("Pre-encoded histories:",len(H_encoded),"| batch size:",BATCH)


train sessions 60104 candidate rows 5360982
val sessions 12880 candidate rows 1156791
test sessions 12880 candidate rows 1147673
Pre-encoded histories: 85864 | batch size: 32


In [12]:
# ============================================================
# HYBRID NCF-LSTM MODEL
# ============================================================

class HybridNCFLSTM(nn.Module):

    def __init__(
        self,
        n_items,
        n_feat,
        attribute_cardinalities
    ):
        super().__init__()

        self.item = nn.Embedding(
            n_items, EMB, padding_idx=0
        )

        self.action = nn.Embedding(
            5, EMB, padding_idx=0
        )

        self.time = nn.Linear(1, EMB)

        self.metadata_embs = nn.ModuleList([
            nn.Embedding(n, EMB, padding_idx=0)
            for n in attribute_cardinalities
        ])

        self.lstm = nn.LSTM(
            EMB,
            HIDDEN,
            num_layers=2,
            batch_first=True,
            dropout=0.3
        )

        self.att = nn.Linear(HIDDEN, 1)

        self.candidate_proj = nn.Linear(
            EMB, HIDDEN
        )

        self.numeric = nn.Sequential(
            nn.Linear(n_feat, HIDDEN),
            nn.ReLU()
        )

        self.head = nn.Sequential(
            nn.Linear(3 * HIDDEN, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.3),

            nn.Linear(64, 4)
        )

    def forward(self, b):

        w = b["hist_wid"]
        a = b["hist_act"]
        g = b["hist_gap"]

        mask = w.ne(0)

        z = (
            self.item(w)
            + self.action(a)
            + self.time(g.unsqueeze(-1))
        )

        z = z * mask.unsqueeze(-1)

        lengths = mask.sum(1).clamp(min=1).cpu()

        packed = nn.utils.rnn.pack_padded_sequence(
            z,
            lengths,
            batch_first=True,
            enforce_sorted=False
        )

        enc, _ = self.lstm(packed)

        enc, _ = nn.utils.rnn.pad_packed_sequence(
            enc,
            batch_first=True,
            total_length=w.size(1)
        )

        att = self.att(enc).squeeze(-1)

        att = att.masked_fill(~mask, -1e9)

        att = torch.where(
            mask.any(1, keepdim=True),
            att,
            torch.zeros_like(att)
        )

        alpha = torch.softmax(att, dim=1) * mask.float()

        alpha = alpha / alpha.sum(
            1, keepdim=True
        ).clamp_min(1e-8)

        user = (
            enc * alpha.unsqueeze(-1)
        ).sum(1)

        # Candidate embedding
        cand_emb = self.item(b["cand"])

        for j, emb in enumerate(self.metadata_embs):
            cand_emb = (
                cand_emb
                + emb(b["attrs"][..., j])
            )

        c = self.candidate_proj(cand_emb)
        f = self.numeric(b["feat"])

        h = torch.cat([
            user[:, None, :].expand_as(c),
            c,
            f
        ], dim=-1)

        # Only genuine candidates enter BatchNorm.
        valid = b["label"].ne(-100)

        output = h.new_zeros(
            (*valid.shape, 4)
        )

        output[valid] = self.head(h[valid])

        return output


model = HybridNCFLSTM(
    len(vocab) + 2,
    len(FEATURES),
    [attr_card[c] for c in ATTR_COLS]
    if USE_METADATA else []
).to(DEVICE)

print("Metadata enabled:", USE_METADATA)

print(
    "Trainable parameters:",
    f"{sum(p.numel() for p in model.parameters()):,}"
)

Metadata enabled: True
Trainable parameters: 39,847,941


In [13]:
# ============================================================
# TRAINING AND VALIDATION
# ============================================================

def score_from_logits(logits):

    values = torch.tensor(
        [0., 1., 2., 3.],
        device=logits.device
    )

    probs = torch.softmax(logits, dim=-1)

    return (probs * values).sum(-1)


def evaluate(split, return_predictions=False):

    model.eval()

    scores = []
    indices = []

    correct = Counter()
    total = Counter()

    with torch.no_grad():

        for batch, ixs, _ in loader(
            split, full_slate=True
        ):

            b = {
                k: v.to(DEVICE)
                for k, v in batch.items()
            }

            logits = model(b)

            ss = score_from_logits(
                logits
            ).cpu().numpy()

            pred = logits.argmax(-1).cpu().numpy()
            y = batch["label"].numpy()

            for j, ix in enumerate(ixs):

                n = len(ix)

                scores.extend(ss[j, :n])
                indices.extend(ix)

                for true, guess in zip(
                    y[j, :n], pred[j, :n]
                ):
                    total[int(true)] += 1
                    correct[int(true)] += int(true == guess)

    d = (
        df.iloc[indices][
            ["session_id", "wid", "label", "split"]
        ]
        .copy()
        .reset_index(drop=True)
    )

    sc = np.asarray(scores)

    assert len(d) == len(sc)
    assert np.isfinite(sc).all()

    m = rank_metrics(d, sc)

    m.update({
        f"class_{k}_recall":
        correct[k] / total[k] if total[k] else np.nan
        for k in range(4)
    })

    return (m, d, sc) if return_predictions else m


optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-5
)

best = -np.inf
log = []
bad = 0
patience = 3

CHECKPOINT = (
    OUT / (
        "best_lstm_ncf_4emb.pt"
        if USE_METADATA
        else "best_lstm_ncf_id_only.pt"
    )
)

for epoch in range(1, EPOCHS + 1):

    model.train()

    loss_sum = 0.
    n = 0
    t0 = time.time()

    for batch, _, _ in loader(
        "train", shuffle=True
    ):

        b = {
            k: v.to(DEVICE)
            for k, v in batch.items()
        }

        optimizer.zero_grad(set_to_none=True)

        logits = model(b)

        loss = nn.functional.cross_entropy(
            logits.reshape(-1, 4),
            b["label"].reshape(-1),
            ignore_index=-100
        )

        loss.backward()

        nn.utils.clip_grad_norm_(
            model.parameters(), 1.0
        )

        optimizer.step()

        valid = (
            b["label"] != -100
        ).sum().item()

        loss_sum += loss.item() * valid
        n += valid

    vm = evaluate("val")

    row = {
        "epoch": epoch,
        "train_ce": loss_sum / max(1, n),
        **{
            "val_" + k: v
            for k, v in vm.items()
        }
    }

    log.append(row)

    print(
        "Epoch:", epoch,
        "| Train CE:", round(row["train_ce"], 5),
        "| Val NDCG@10:", round(vm["ndcg@10"], 5),
        "| Minutes:", round((time.time()-t0)/60, 2)
    )

    if vm["ndcg@10"] > best + 1e-6:

        best = vm["ndcg@10"]
        bad = 0

        torch.save({
            "model": model.state_dict(),
            "epoch": epoch,
            "features": FEATURES,
            "vocab_size": len(vocab),
            "metadata": USE_METADATA
        }, CHECKPOINT)

    else:

        bad += 1

        if bad >= patience:
            print("Early stopping")
            break

pd.DataFrame(log).to_csv(
    OUT / "baseline_training_history.csv",
    index=False
)

print("Best validation NDCG@10:", best)

Epoch: 1 | Train CE: 0.30052 | Val NDCG@10: 0.30361 | Minutes: 2.06
Epoch: 2 | Train CE: 0.22787 | Val NDCG@10: 0.33242 | Minutes: 2.06
Epoch: 3 | Train CE: 0.22248 | Val NDCG@10: 0.34651 | Minutes: 2.05
Epoch: 4 | Train CE: 0.21879 | Val NDCG@10: 0.35294 | Minutes: 2.06
Epoch: 5 | Train CE: 0.21421 | Val NDCG@10: 0.35674 | Minutes: 2.04
Epoch: 6 | Train CE: 0.20855 | Val NDCG@10: 0.35669 | Minutes: 2.04
Epoch: 7 | Train CE: 0.20207 | Val NDCG@10: 0.34756 | Minutes: 2.02
Epoch: 8 | Train CE: 0.19457 | Val NDCG@10: 0.34128 | Minutes: 2.01
Early stopping
Best validation NDCG@10: 0.3567396523319721


In [14]:
# ============================================================
# FINAL RANKING EVALUATION
# ============================================================

checkpoint = torch.load(
    CHECKPOINT,
    map_location=DEVICE,
    weights_only=False
)

model.load_state_dict(checkpoint["model"])

results = []

bm25_col = next(
    c for c in df.columns
    if "bm25_query_name" in c.lower()
)

bm25_lookup = df[["session_id", "wid", bm25_col]].drop_duplicates(["session_id", "wid"])

for s in ["train", "val", "test"]:

    nn_metrics, d, nn_scores = evaluate(
        s, return_predictions=True
    )

    # DISTINCT-USER POPULARITY + ORDER TIE-BREAK

    pop = np.empty(len(d), dtype=np.float64)

    for j, (sid, wid) in enumerate(
        zip(d["session_id"], d["wid"])
    ):

        sid = int(sid)
        wid = norm_id(wid)

        n_users = pop_product_users.get(wid, 0)
        n_orders = pop_product_ord.get(wid, 0)

        # Leave-self-out for training sessions only
        if s == "train":
            n_users -= int((sid, wid) in own_products)
            n_orders -= own_orders.get((sid, wid), 0)

        pop[j] = n_users + 0.001 * n_orders
    # Lookup once via original row order when possible; preserve key validation.
    d = d.merge(bm25_lookup, on=["session_id", "wid"], how="left", validate="one_to_one")

    bm25 = (
        d[bm25_col]
        .fillna(0)
        .to_numpy()
    )

    for name, sc in [
        ("popularity", pop),
        ("bm25", bm25),
        ("lstm_ncf_ce", nn_scores)
    ]:

        results.append({
            "model": name,
            "split": s,
            **rank_metrics(d, sc)
        })

    if s == "test":

        pred = d[
            ["session_id", "wid", "label", "split"]
        ].copy()

        pred["score_lstm_ncf"] = nn_scores
        pred["score_popularity"] = pop
        pred["score_bm25"] = bm25

        pred.to_parquet(
            OUT / "test_predictions_baselines.parquet",
            index=False
        )

        print(
            "Test classifier recall:",
            {
                k: v for k, v in nn_metrics.items()
                if "recall" in k
            }
        )

res = pd.DataFrame(results)

print("\n=== ALL RANKING RESULTS ===")
display(res)

print("\n=== TEST SET PERFORMANCE ===")

display(
    res.loc[
        res["split"] == "test",
        [
            "model",
            "ndcg@5",
            "ndcg@10",
            "mrr",
            "purchase_hit@10",
            "n_searches"
        ]
    ].round(5)
)

res.to_csv(
    OUT / "results_baselines.csv",
    index=False
)

print("\nOutputs saved to:", OUT)

Test classifier recall: {'class_0_recall': 0.9995666272745394, 'class_1_recall': 0.006354009077155824, 'class_2_recall': 0.004108182129407737, 'class_3_recall': 0.002560819462227913}

=== ALL RANKING RESULTS ===


,model,split,ndcg@5,ndcg@10,mrr,purchase_hit@10,n_purchase_searches,n_searches
0,popularity,train,0.286632,0.335081,0.334187,0.573660,7107,60104
1,bm25,train,0.067259,0.092805,0.109232,0.150415,7107,60104
2,lstm_ncf_ce,train,0.409062,0.455479,0.463880,0.714507,7107,60104
3,popularity,val,0.276016,0.324238,0.322545,0.568615,1523,12880
4,bm25,val,0.066023,0.091232,0.107293,0.156927,1523,12880
5,lstm_ncf_ce,val,0.311519,0.356740,0.368573,0.571898,1523,12880
6,popularity,test,0.278427,0.325704,0.323188,0.562049,1523,12880
7,bm25,test,0.064983,0.089988,0.107077,0.151018,1523,12880
8,lstm_ncf_ce,test,0.308474,0.352815,0.360766,0.550230,1523,12880



=== TEST SET PERFORMANCE ===


,model,ndcg@5,ndcg@10,mrr,purchase_hit@10,n_searches
6,popularity,0.27843,0.32570,0.32319,0.56205,12880
7,bm25,0.06498,0.08999,0.10708,0.15102,12880
8,lstm_ncf_ce,0.30847,0.35281,0.36077,0.55023,12880



Outputs saved to: /content/drive/MyDrive/bt4222/JDsearch/step2_baselines
